# Paper results: Marker-CXR fine-tuning of MedGemma-1.5-4B

Reads the predictions that `run_arms.sh` writes to `../runs/` (all 8 arms, internal and
VinDr-CXR). Writes tables (CSV) and figures (PNG + PDF) to `../runs/paper/`.

| paper | here |
|---|---|
| Table 1 | Tables 1a + 1b (without the `full` row) |
| Fig. 3 | Table 3 (`fig_table3_differences`) |
| Fig. 4 | scaling figure (`fig_marker_scaling`) |

Tables 2, 4, 5, 6 and S1 are extra results, not in the paper.

Every number has a 95% confidence interval from a **paired cluster bootstrap**
(100,000 draws). It resamples test images, not single rows, because several
questions share one image. Every arm is scored on the same resampled images, so
a difference between two arms gets its own interval. Each difference also gets a
two-sided p-value from the same draws. p is Holm-corrected within each table.
Raw p is in the `*_long.csv` files. Runs on CPU in a few minutes.

**Arms**

| arm | what it is |
|---|---|
| `baseline` | MedGemma-1.5-4B zero-shot (reasoning prompt, its best setting) |
| `grpo_box_base_reasoning` | GRPO straight from the base model, no SFT (bbox reward: IoU + center distance; reasoning prompts) |
| `no_marker` | SFT on Marker-CXR without the 16 marker question types |
| `no_reasoning` | SFT on all of Marker-CXR, direct prompts (main SFT model) |
| `full` | same as `no_reasoning` but trained and prompted with reasoning traces (Tables 1a, 1b only, not in the paper) |
| `grpo_box_no_reasoning` | `no_reasoning` + GRPO with the same bbox reward (best model) |
| `marker25`, `marker50` | `no_marker` + 25% / 50% of the marker rows (scaling only) |

**Metrics.** Box questions: mean IoU, IoU ≥ 0.5 (share of boxes), miss
(share with IoU = 0), and area ratio = median of predicted / true box area
(1.00 = right size, above 1 = too big). Non-box questions: exact match,
averaged over question types so each type counts the same. VinDr-CXR is the
external test set (never trained on): finding boxes, plus yes/no questions.

In [ ]:
import importlib
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "src")
sys.path.insert(0, "../vindr")
from bootstrap_ci import boot_p, bootstrap, fmt_p, holm, wmean, wmedian, wtypemean
from paper_charts import Charts

RUNS = Path("../runs")     # run_arms.sh output
VINDR = RUNS / "vindr"
OUT = RUNS / "paper"
OUT.mkdir(parents=True, exist_ok=True)

MAIN = ["baseline", "grpo_box_base_reasoning", "no_marker", "no_reasoning", "full",
        "grpo_box_no_reasoning"]                                   # Tables 1a, 1b
SHOW = [a for a in MAIN if a != "full"]                            # every other table
SCALING = ["no_marker", "marker25", "marker50", "no_reasoning"]   # 0 / 25 / 50 / 100% marker data
ARMS = list(dict.fromkeys(MAIN + SCALING))
N_BOOT, SEED = 100000, 0   # with 10k draws, the smallest Holm p would be m/10,001 (0.003 for 27 tests)

BOX = ["anatomical_region_localization", "finding_localization_mscxr",
       "finding_localization_mscxr_region_conditioned", "marker_localization",
       "marker_localization_region_conditioned", "marker_localization_quadrant_conditioned"]
# The 16 types no_marker leaves out of training (all questions about markers)
MARKER_TYPES = {"marker_existence", "marker_counting", "marker_grounding", "marker_grounding_discriminative",
                "marker_localization", "marker_localization_region_conditioned",
                "marker_localization_quadrant_conditioned", "marker_localization_negative",
                "marker_count_region_conditioned", "marker_count_quadrant_conditioned",
                "region_conditioned_marker_counting", "coordinate_marker_existence", "coordinate_to_label",
                "coordinate_to_label_negative", "category_discrimination", "region_to_object"}
# "Where is X?" questions where X is not in the image; the right answer says so
NEGATIVE = ["marker_localization_negative", "finding_localization_mscxr_negative", "coordinate_to_label_negative"]
SAYS_ABSENT = re.compile(r"\b(?:no|not|none|absent)\b")  # lenient check on the final answer
# VinDr labels that Marker-CXR trains on (specific_label_existence); the rest are unseen
SEEN = {"atelectasis", "cardiomegaly", "consolidation", "lung_opacity",
        "pleural_effusion", "pneumonia", "pneumothorax"}

# VinDr question text -> label (module name starts with a digit, so plain `import` can't reach it)
PHRASE_TO_LABEL = {v: k for k, v in importlib.import_module("02_build_vindr_vqa").LABEL_PHRASE.items()}
_EXIST_RE = re.compile(r"Is there evidence of (.+?) in this image\?")
_LOC_RE = re.compile(r"Where is the (.+?)\? Give its full bounding box")


def label_of(question):
    m = _EXIST_RE.search(question) or _LOC_RE.search(question)
    return PHRASE_TO_LABEL.get(m.group(1), m.group(1)) if m else "(finding_existence)"

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

## Load and align predictions

Arms list their rows in different orders, and box questions carry
different prompt wrappers (reasoning vs direct). Rows are matched on
image + question type + the plain question inside the wrapper.

In [ ]:
_QUERY = re.compile(r"Query:\n(.*?)(?: Don't give a final answer| Answer:)", re.S)


def plain_question(q):
    m = _QUERY.search(q)
    return (m.group(1) if m else q).strip()


def load(path):
    df = pd.read_csv(path)
    key = df["image_path"] + "|" + df["question_type"] + "|" + df["question"].map(plain_question)
    df.index = pd.MultiIndex.from_arrays([key, key.groupby(key).cumcount()], names=["key", "n"])
    return df


def internal_path(arm):
    return RUNS / arm / "eval" / ("" if arm == "baseline" else "finetuned") / "predictions.csv"


internal = {a: load(internal_path(a)) for a in ARMS}
vindr = {a: load(VINDR / a / "predictions.csv") for a in ARMS}
for name, data in [("internal", internal), ("VinDr", vindr)]:
    ref = data[ARMS[0]].index
    assert all(set(d.index) == set(ref) for d in data.values()), f"{name}: arms scored different rows"
    for a in ARMS:
        data[a] = data[a].loc[ref]  # one row order for every arm
    print(f"{name}: {len(ref)} rows, {data[ARMS[0]]['image_path'].nunique()} images, arms aligned")

base_int, base_vin = internal[ARMS[0]], vindr[ARMS[0]]
base_vin = base_vin.assign(label=base_vin["question"].map(label_of))


def wide(data, col, rows):
    """rows x arms array of one per-row column."""
    return np.column_stack([data[a].loc[rows, col].astype(float).to_numpy() for a in ARMS])

## Bootstrap every metric

One long table `res` (point estimate + 95% CI per dataset / group / metric /
arm) and the raw draws, kept for the paired differences.

In [ ]:
res_rows, DRAWS = [], {}


def score(dataset, group, metric, stat, X, images, **kw):
    point, draws = bootstrap(stat, X, images, n_boot=N_BOOT, seed=SEED, **kw)
    lo, hi = np.percentile(draws, [2.5, 97.5], axis=0)
    DRAWS[(dataset, group, metric)] = draws
    for a, p, l, h in zip(ARMS, point, lo, hi):
        res_rows.append(dict(dataset=dataset, group=group, metric=metric, arm=a, point=p, lo=l, hi=h, n=len(X)))


def score_boxes(dataset, data, group, rows):
    images = data[ARMS[0]].loc[rows, "image_path"].to_numpy()
    iou = wide(data, "iou", rows)
    score(dataset, group, "IoU", wmean, iou, images)
    score(dataset, group, "IoU>=0.5", wmean, (iou >= 0.5).astype(float), images)
    score(dataset, group, "miss", wmean, (iou == 0).astype(float), images)
    score(dataset, group, "area ratio", wmedian, wide(data, "area_ratio", rows), images)


qt = base_int["question_type"]
for group, mask in [("all boxes", qt.isin(BOX)),
                    ("non-marker boxes", qt.isin(BOX) & ~qt.isin(MARKER_TYPES)),
                    ("anatomy", qt == "anatomical_region_localization"),
                    ("finding", qt.str.startswith("finding_localization_mscxr") & qt.isin(BOX)),
                    ("marker", qt.str.startswith("marker_localization") & qt.isin(BOX))]:
    score_boxes("internal", internal, group, base_int.index[mask])
for t in BOX:  # each box type on its own (supplementary table)
    score_boxes("internal", internal, t, base_int.index[qt == t])

# Exact match of every non-box type on its own (existence types: = accuracy)
NONBOX_TYPES = sorted(set(qt) - set(BOX), key=lambda t: (t in MARKER_TYPES, t))
for t in NONBOX_TYPES:
    rows = base_int.index[qt == t]
    score("internal", t, "exact match", wmean, wide(internal, "correct", rows), base_int.loc[rows, "image_path"].to_numpy())

for t in NEGATIVE:
    rows = base_int.index[qt == t]
    images = base_int.loc[rows, "image_path"].to_numpy()
    absent = np.column_stack([internal[a].loc[rows, "prediction_normalized"].astype(str).str.contains(SAYS_ABSENT)
                              .to_numpy(float) for a in ARMS])
    score("internal", t, "says absent", wmean, absent, images)

for group, mask in [("non-box", ~qt.isin(BOX)), ("non-box, non-marker types", ~qt.isin(BOX) & ~qt.isin(MARKER_TYPES))]:
    rows = base_int.index[mask]
    score("internal", group, "exact match", wtypemean, wide(internal, "correct", rows),
          base_int.loc[rows, "image_path"].to_numpy(), types=base_int.loc[rows, "question_type"].to_numpy())

vqt = base_vin["question_type"]
loc = vqt == "finding_localization_mscxr"
for group, mask in [("all boxes", loc), ("seen findings", loc & base_vin["label"].isin(SEEN)),
                    ("unseen findings", loc & ~base_vin["label"].isin(SEEN))]:
    score_boxes("VinDr", vindr, group, base_vin.index[mask])
label_q = vqt == "specific_label_existence"
VINDR_YN = [("finding_existence", vqt == "finding_existence"), ("specific_label_existence", label_q),
            ("specific_label_existence, seen labels", label_q & base_vin["label"].isin(SEEN)),
            ("specific_label_existence, unseen labels", label_q & ~base_vin["label"].isin(SEEN))]
for group, mask in VINDR_YN:
    rows = base_vin.index[mask]
    score("VinDr", group, "accuracy", wmean, wide(vindr, "correct", rows), base_vin.loc[rows, "image_path"].to_numpy())

# Yes/no answer shares on existence questions: % answered "yes", % "no"
# (the rest is neither, e.g. a long free-text answer). TRUE_YES = share of
# questions whose correct answer is "yes" (same for every arm, no CI).
EXISTENCE = ["marker_existence", "finding_existence", "specific_label_existence",
             "coordinate_marker_existence", "marker_grounding_discriminative"]
TRUE_YES = {}


def score_yes_no(dataset, data, group, rows):
    images = data[ARMS[0]].loc[rows, "image_path"].to_numpy()
    for metric, value in [("answered yes", "yes"), ("answered no", "no")]:
        X = np.column_stack([(data[a].loc[rows, "prediction_normalized"] == value).to_numpy(float) for a in ARMS])
        score(dataset, group, metric, wmean, X, images)
    TRUE_YES[(dataset, group)] = (data[ARMS[0]].loc[rows, "reference_normalized"] == "yes").mean()


for t in EXISTENCE:
    score_yes_no("internal", internal, t, base_int.index[qt == t])
for group, mask in VINDR_YN:
    score_yes_no("VinDr", vindr, group, base_vin.index[mask])

res = pd.DataFrame(res_rows)
res.to_csv(OUT / "metrics_with_ci.csv", index=False)
print(len(res), "rows ->", OUT / "metrics_with_ci.csv")

## Helpers to print "point [low, high]"

In [ ]:
PERCENT = {"miss", "answered yes", "answered no"}


def fmt(metric, p, lo, hi, signed=False):
    s = "+" if signed else ""
    if metric in PERCENT:
        return f"{p * 100:{s}.1f}% [{lo * 100:{s}.1f}, {hi * 100:{s}.1f}]"
    if metric == "area ratio":
        return f"{p:{s}.2f} [{lo:{s}.2f}, {hi:{s}.2f}]"
    return f"{p:{s}.3f} [{lo:{s}.3f}, {hi:{s}.3f}]"


def table(dataset, columns, arms):
    """columns: list of (header, group, metric). Rows = arms."""
    out = {}
    for header, group, metric in columns:
        sel = res[(res.dataset == dataset) & (res.group == group) & (res.metric == metric)].set_index("arm")
        out[header] = [fmt(metric, *sel.loc[a, ["point", "lo", "hi"]]) for a in arms]
    return pd.DataFrame(out, index=arms)


LONG = {}  # diff_table name -> long table (raw p and Holm p), used by charts.forest()


def diff_table(pairs, columns, name):
    """Paired differences (arm minus reference): 95% CI and Holm p-value per cell.
    pairs: list of (label, arm, reference); columns: list of (header, dataset,
    group, metric). Holm correction over every cell of this table. Writes the
    long numeric form (raw p and Holm p) to <OUT>/<name>_long.csv; returns the
    display table."""
    rows = []
    for header, dataset, group, metric in columns:
        d = DRAWS[(dataset, group, metric)]
        sel = res[(res.dataset == dataset) & (res.group == group) & (res.metric == metric)].set_index("arm")["point"]
        for label, a, b in pairs:
            dd = d[:, ARMS.index(a)] - d[:, ARMS.index(b)]
            lo, hi = np.percentile(dd, [2.5, 97.5])
            pt = sel[a] - sel[b]
            rows.append(dict(comparison=label, arm=a, reference=b, metric_name=header, metric=metric,
                             diff=pt, lo=lo, hi=hi, p=boot_p(dd, pt)))
    long = pd.DataFrame(rows)
    long["p_holm"] = holm(long["p"])
    long.to_csv(OUT / f"{name}_long.csv", index=False)
    LONG[name] = long
    long["cell"] = [fmt(r.metric, r.diff, r.lo, r.hi, signed=True) + "  " + fmt_p(r.p_holm, "p") for r in long.itertuples()]
    table = long.pivot(index="comparison", columns="metric_name", values="cell")
    table = table.loc[[p[0] for p in pairs], [c[0] for c in columns]]
    table.to_csv(OUT / f"{name}.csv")
    changed = long[(long.p < 0.05) != (long.p_holm < 0.05)]
    print(f"Holm over {len(long)} tests: {len(changed)} cell(s) change significance at 0.05"
          + "".join(f"\n  {r.comparison} / {r.metric_name}: p={fmt_p(r.p)} -> Holm {fmt_p(r.p_holm)}"
                    for r in changed.itertuples()))
    return table

## Charts

Chart code is in `src/paper_charts.py` (one panel per metric, dot + 95% CI;
area ratio on a log axis with a dashed line at 1 = right size).

In [ ]:
NAMES = {"baseline": "Zero-shot", "grpo_box_base_reasoning": "GRPO, no SFT",
         "no_marker": "SFT, no marker data", "no_reasoning": "SFT",
         "full": "SFT + reasoning", "grpo_box_no_reasoning": "SFT + GRPO",
         "marker25": "SFT, 25% marker data", "marker50": "SFT, 50% marker data"}
charts = Charts(res, DRAWS, ARMS, NAMES, OUT, percent=PERCENT)

## Table 1a — internal test set: findings and anatomy

Finding boxes (MS-CXR, 2 types: plain and region-conditioned) and anatomy
boxes (Chest ImaGenome regions), plus yes/no finding existence. Only 124
internal finding-box rows, hence the wide CIs; VinDr (Tables 2a, 2b) has 1,594.

In [ ]:
FINDING_COLS = [("finding IoU", "finding", "IoU"), ("finding IoU≥0.5", "finding", "IoU>=0.5"),
                ("finding miss", "finding", "miss"), ("finding area ratio", "finding", "area ratio"),
                ("anatomy IoU", "anatomy", "IoU"), ("anatomy IoU≥0.5", "anatomy", "IoU>=0.5"),
                ("anatomy area ratio", "anatomy", "area ratio"),
                ("finding existence acc", "finding_existence", "exact match")]
t1a = table("internal", FINDING_COLS, MAIN)
t1a.to_csv(OUT / "table1a_internal_findings.csv")
t1a

In [ ]:
charts.ci_panels("internal", FINDING_COLS, MAIN, ncols=4, fname="fig_table1a_internal_findings")

## Table 1b — internal test set: markers

Marker boxes (3 types: plain, region-conditioned, quadrant-conditioned) and
yes/no marker existence. `no_marker` never trains on marker questions, so
its row shows the base model's marker skill after SFT on everything else.

In [ ]:
MARKER_COLS = [("marker IoU", "marker", "IoU"), ("marker IoU≥0.5", "marker", "IoU>=0.5"),
               ("marker miss", "marker", "miss"), ("marker area ratio", "marker", "area ratio"),
               ("marker existence acc", "marker_existence", "exact match")]
t1b = table("internal", MARKER_COLS, MAIN)
t1b.to_csv(OUT / "table1b_internal_markers.csv")
t1b

In [ ]:
charts.ci_panels("internal", MARKER_COLS, MAIN, ncols=5, fname="fig_table1b_internal_markers")

## Table 2a — external test set (VinDr-CXR, never trained on), seen findings

Findings Marker-CXR trains on (7 labels). Boxes, then the label yes/no
question for the same labels. All-box results are in Table 3 and Table S1;
"any finding" existence accuracy is in Table S1.

In [ ]:
def vindr_cols(kind):
    g = f"{kind} findings"
    return [(f"{kind} IoU", g, "IoU"), (f"{kind} IoU≥0.5", g, "IoU>=0.5"),
            (f"{kind} miss", g, "miss"), (f"{kind} area ratio", g, "area ratio"),
            (f"{kind} label existence acc", f"specific_label_existence, {kind} labels", "accuracy")]


SEEN_COLS = vindr_cols("seen")
t2a = table("VinDr", SEEN_COLS, SHOW)
t2a.to_csv(OUT / "table2a_vindr_seen.csv")
t2a

In [ ]:
charts.ci_panels("VinDr", SEEN_COLS, SHOW, ncols=5, fname="fig_table2a_vindr_seen")

## Table 2b — external test set, unseen findings

VinDr findings Marker-CXR never trains on.

In [ ]:
UNSEEN_COLS = vindr_cols("unseen")
t2b = table("VinDr", UNSEEN_COLS, SHOW)
t2b.to_csv(OUT / "table2b_vindr_unseen.csv")
t2b

In [ ]:
charts.ci_panels("VinDr", UNSEEN_COLS, SHOW, ncols=5, fname="fig_table2b_vindr_unseen")

## Table 3 — what each step adds (paired differences, 95% CI)

p = two-sided paired bootstrap p-value, Holm-corrected over all cells of this
table (raw p in `table3_differences_long.csv`; the printout lists cells where
the correction changes significance at 0.05).
Only non-marker metrics: marker results are in Table 1, and `no_marker`
never trained on marker questions. Area ratio is tested but not plotted.

In [ ]:
PAIRS = [("SFT vs zero-shot (no_reasoning − baseline)", "no_reasoning", "baseline"),
         ("+ marker data (no_reasoning − no_marker)", "no_reasoning", "no_marker"),
         ("+ GRPO (grpo_box_no_reasoning − no_reasoning)", "grpo_box_no_reasoning", "no_reasoning")]
DIFF_COLS = [("internal anatomy IoU", "internal", "anatomy", "IoU"),
             ("internal finding IoU", "internal", "finding", "IoU"),
             ("internal finding existence acc", "internal", "finding_existence", "exact match"),
             ("internal label existence acc", "internal", "specific_label_existence", "exact match"),
             ("VinDr box IoU", "VinDr", "all boxes", "IoU"),
             ("VinDr box IoU≥0.5", "VinDr", "all boxes", "IoU>=0.5"),
             ("VinDr area ratio", "VinDr", "all boxes", "area ratio"),  # tested, not plotted
             ("VinDr finding existence acc", "VinDr", "finding_existence", "accuracy"),
             ("VinDr label existence acc", "VinDr", "specific_label_existence", "accuracy")]
t3 = diff_table(PAIRS, DIFF_COLS, "table3_differences")
t3.T

In [ ]:
charts.forest(PAIRS, DIFF_COLS, "fig_table3_differences", LONG["table3_differences"])

## Table 4 — marker-data scaling

All four arms train on the same 39,134 non-marker rows (including the same
1,146 radiologist-drawn MS-CXR finding boxes). Only the amount of generated
marker data changes: all questions of a random 25% / 50% of the 7,206 marker
images (the 25% images are a subset of the 50% ones). These counts and `MARKER_ROWS` below
are the paper's numbers. Change them if your rebuilt data has other counts.

In [ ]:
MARKER_ROWS = {"no_marker": 0, "marker25": 20964, "marker50": 42048, "no_reasoning": 85533}
SCALE_COLS = [("VinDr box IoU", "VinDr", "all boxes", "IoU"), ("VinDr box IoU≥0.5", "VinDr", "all boxes", "IoU>=0.5"),
              ("VinDr box area ratio", "VinDr", "all boxes", "area ratio"),
              ("internal anatomy IoU", "internal", "anatomy", "IoU"),
              ("internal finding IoU", "internal", "finding", "IoU"),
              ("internal marker IoU", "internal", "marker", "IoU")]
t4 = pd.concat([table(ds, [(h, g, m)], SCALING) for h, ds, g, m in SCALE_COLS], axis=1)
t4.insert(0, "marker train rows", [f"{MARKER_ROWS[a]:,}" for a in SCALING])
t4.index = ["0% (no_marker)", "25% (marker25)", "50% (marker50)", "100% (no_reasoning)"]
t4.to_csv(OUT / "table4_scaling.csv")
t4

In [ ]:
# Step-to-step gains, paired (p Holm-corrected over this table)
STEPS = [("0% → 25%", "marker25", "no_marker"), ("25% → 50%", "marker50", "marker25"),
         ("50% → 100%", "no_reasoning", "marker50"), ("0% → 100%", "no_reasoning", "no_marker")]
t4b = diff_table(STEPS, SCALE_COLS, "table4b_scaling_steps")
t4b

Trend test: one number per metric for "more marker data → better". Slope of
a least-squares line through the 4 points (x = share of marker data, 0 to 1),
so the slope is the fitted change from no marker data to all of it. CI and
p-value from the same paired draws; p Holm-corrected over the 6 slopes.

In [ ]:
x = np.array([0, 0.25, 0.5, 1.0])
rows = []
for header, dataset, group, metric in SCALE_COLS:
    d = DRAWS[(dataset, group, metric)][:, [ARMS.index(a) for a in SCALING]]
    pts = res[(res.dataset == dataset) & (res.group == group) & (res.metric == metric)].set_index("arm").loc[SCALING, "point"].to_numpy()
    slope = lambda y: ((x - x.mean()) * (y - y.mean(axis=-1, keepdims=True))).sum(axis=-1) / ((x - x.mean()) ** 2).sum()
    draws, pt = slope(d), slope(pts)
    lo, hi = np.percentile(draws, [2.5, 97.5])
    rows.append(dict(metric=header, slope=pt, lo=lo, hi=hi, p=boot_p(draws, pt)))
t4c = pd.DataFrame(rows)
t4c["p_holm"] = holm(t4c["p"])
t4c.to_csv(OUT / "table4c_scaling_trend.csv", index=False)
t4c.assign(slope=[fmt(m, s, l, h, signed=True) for m, s, l, h in
                  zip([c[3] for c in SCALE_COLS], t4c.slope, t4c.lo, t4c.hi)],
           p=t4c.p_holm.map(fmt_p)).drop(columns=["lo", "hi", "p_holm"]).set_index("metric")  # p = Holm

In [ ]:
charts.forest(STEPS, SCALE_COLS, "fig_table4b_scaling_steps", LONG["table4b_scaling_steps"], title_of=lambda label, a, b: f"marker data {label}")

## Figure — marker-data scaling with 95% CIs

VinDr finding boxes only, one panel per measure (separate y-axes, never two
scales on one axis). x = number of generated marker training rows. Internal
results are in Table 4.

In [ ]:
PANELS = [("VinDr", "all boxes", "IoU>=0.5", "VinDr finding boxes\nIoU ≥ 0.5"),
          ("VinDr", "all boxes", "IoU", "VinDr finding boxes\nmean IoU"),
          ("VinDr", "all boxes", "area ratio", "VinDr finding boxes\narea ratio (1 = right size)")]
charts.scaling(PANELS, SCALING, x=np.array([MARKER_ROWS[a] for a in SCALING]) / 1000,
               xticklabels=["0\n0%", "21k\n25%", "42k\n50%", "86k\n100%"],
               xlabel="generated marker training rows (share of all marker data)", fname="fig_marker_scaling")

## Table 5 — yes/no answers on existence questions

How often each arm answers "yes" and "no", next to the true share of each
answer ("true share"). Internal: marker and finding existence only (the other
yes/no types are in Table S1).
A yes-share far from the true share means a biased model (e.g. one that says
"no" too often misses real findings). Yes + no can fall short of 100% when
an answer is neither (the zero-shot model sometimes writes free text).
VinDr label questions are also split into labels Marker-CXR trains on
(seen) and the rest (unseen).

In [ ]:
YN_GROUPS = [("internal", t) for t in ["marker_existence", "finding_existence"]] + [("VinDr", g) for g in [
    "finding_existence", "specific_label_existence", "specific_label_existence, seen labels",
    "specific_label_existence, unseen labels"]]
rows = []
for dataset, group in YN_GROUPS:
    for metric in ["answered yes", "answered no"]:
        sel = res[(res.dataset == dataset) & (res.group == group) & (res.metric == metric)].set_index("arm")
        rows.append({"dataset": dataset, "question type": group, "answer": metric.split()[1],
                     # true share of this answer: "yes" rows get the true yes share, "no" rows the rest
                     "true share": f"{(TRUE_YES[(dataset, group)] if metric == 'answered yes' else 1 - TRUE_YES[(dataset, group)]) * 100:.1f}%",
                     **{a: fmt(metric, *sel.loc[a, ["point", "lo", "hi"]]) for a in SHOW}})
t5 = pd.DataFrame(rows).set_index(["dataset", "question type", "answer"])
t5.to_csv(OUT / "table5_yes_no.csv")
t5

In [ ]:
# Share answered "yes" per arm (%, same 0-100 axis everywhere); dashed line = true share of "yes"
ci_panels_cols = [(f"{ds}: {g.replace('specific_label_existence', 'label existence').replace('_', ' ')}", g, "answered yes")
                  for ds, g in YN_GROUPS]
for ds in ["internal", "VinDr"]:
    cols = [c for c, (d, _) in zip(ci_panels_cols, YN_GROUPS) if d == ds]
    charts.ci_panels(ds, cols, SHOW, ncols=len(cols), fname=f"fig_table5_yes_rate_{ds.lower()}",
                     refs=[TRUE_YES[(ds, g)] * 100 for _, g, _ in cols], xlim=(0, 100))

## Table 6 — localization questions about something not in the image

"Where is the marker?" / "Where is the <finding>?" when there is no marker /
no such finding; the right answer says it is not there (no box). Metric:
says absent = share of answers that contain "no", "not", "none" or "absent"
(checked by hand on the zero-shot finding answers: 161 of the 162 matches
really say absent). Internal test set only. Exact match and
`coordinate_to_label_negative` are in Table S1.

In [ ]:
NEG_COLS = [("marker not in image", "marker_localization_negative", "says absent"),
            ("finding not in image", "finding_localization_mscxr_negative", "says absent")]
t6 = table("internal", NEG_COLS, SHOW).add_suffix(": says absent")
t6.to_csv(OUT / "table6_localization_negatives.csv")
t6

In [ ]:
charts.ci_panels("internal", NEG_COLS, SHOW, ncols=2, fname="fig_table6_localization_negatives", xlim=(0, 1),
                 wspace=0.3, xlabel="says absent (share of answers)")

## Table S1 — every result in detail (not in the paper)

All 8 arms, every question type, every metric, each as value [95% CI].
Rows: internal box results (overall, by group, then each box type), then
every non-box type (exact match; for yes/no types also % answered yes/no;
for negative types also the lenient "says absent"), then VinDr. `n` = test
rows; "true share" = share of correct answers that are "yes" (or "no").
Saved as `tableS1_all_results.csv` (formatted) and `metrics_with_ci.csv`
(numbers).

In [ ]:
S1_ARMS = ["baseline", "grpo_box_base_reasoning", "no_marker", "marker25", "marker50", "no_reasoning", "full", "grpo_box_no_reasoning"]
BOX_GROUPS = ["all boxes", "non-marker boxes", "anatomy", "finding", "marker"]
METRIC_ORDER = ["IoU", "IoU>=0.5", "miss", "area ratio", "exact match", "accuracy", "says absent",
                "answered yes", "answered no"]


def s1_key(r):
    """Sort rows: dataset, then section, then question type, then metric."""
    g = r["group"]
    if r["dataset"] == "internal":
        section = (0 if g in BOX_GROUPS else 1 if g in BOX else 2 if g.startswith("non-box") else 3)
        within = (BOX_GROUPS.index(g) if g in BOX_GROUPS else BOX.index(g) if g in BOX
                  else NONBOX_TYPES.index(g) if g in NONBOX_TYPES else 0)
    else:
        section, within = (0, ["all boxes", "seen findings", "unseen findings"].index(g)) if "findings" in g or g == "all boxes" \
            else (1, [v[0] for v in VINDR_YN].index(g))
    return (r["dataset"] != "internal", section, within, METRIC_ORDER.index(r["metric"]))


combos = res[["dataset", "group", "metric", "n"]].drop_duplicates(["dataset", "group", "metric"])
combos = combos.iloc[sorted(range(len(combos)), key=lambda i: s1_key(combos.iloc[i]))]
rows = []
for c in combos.itertuples():
    sel = res[(res.dataset == c.dataset) & (res.group == c.group) & (res.metric == c.metric)].set_index("arm")
    true = TRUE_YES.get((c.dataset, c.group))
    true_share = "" if true is None or c.metric not in ("answered yes", "answered no") else \
        f"{(true if c.metric == 'answered yes' else 1 - true) * 100:.1f}%"
    rows.append({"dataset": c.dataset, "question type / group": c.group, "metric": c.metric, "n": c.n,
                 "true share": true_share, **{a: fmt(c.metric, *sel.loc[a, ["point", "lo", "hi"]]) for a in S1_ARMS}})
s1 = pd.DataFrame(rows).set_index(["dataset", "question type / group", "metric"])
s1.to_csv(OUT / "tableS1_all_results.csv")
print(len(s1), "rows")
with pd.option_context("display.max_rows", 500):
    display(s1)

In [ ]:
print("saved:", sorted(p.name for p in OUT.iterdir()))